In this nb we build a RAG (Retrieval-Augmented Generation) system using numpy and MistralAi api.

A RAG system combines 2 things:

- a retriever : finds useful information from a database
- a generator : writes the answer using retrieved information

We need RAG because LLMs internal knowledge can be incomplete or incorrect (ex: training cutoff, private information)

RAG allows models to use external knowledge at query time

In [3]:
import numpy as np
from mistralai import Mistral
import os
import string

The infrastructure can be generalised to a lot of use-cases but here we choose to assist customer support.

Our knowledge base is made of basic user information. For this example, we are going to focus on James Turner (CUST002), who has an extended conversation history with the sales team.

In [4]:
documents = [
    "Customer ID: CUST001 | Name: Maria Alvarez | Age: 32 | Subscription Tier: Premium | Joined: 2023-04-10 | Last Payment: 2025-12-01 | Payment Status: On-time | Claims: 2 (Laptop repair Apr 2024 - approved; Smartphone screen Sep 2025 - approved) | Support Notes: Very responsive, prefers email communication.",
    
    "Customer ID: CUST002 | Name: James Turner | Age: 45 | Subscription Tier: Basic | Joined: 2022-10-02 | Last Payment: 2025-11-15 | Payment Status: Late last month but now settled | Claims: 0 | Support Notes: ((2023-06-02) Interested in upgrading next year. ; (2024-05-12) Declined sales team special Premium tier offer. ; (2024-05-17) Complained about Basic tier limitations ; (2025-01-06) Asked for a Premium Tier discount, refused by sales team ",
    
    "Customer ID: CUST003 | Name: Amina Khalid | Age: 27 | Subscription Tier: Standard | Joined: 2024-06-20 | Last Payment: 2025-12-03 | Payment Status: On-time | Claims: 1 (Water damage claim Oct 2025 - under review) | Support Notes: Frequent traveler, requests SMS notifications.",
    
    "Customer ID: CUST004 | Name: Luca Rossi | Age: 38 | Subscription Tier: Premium | Joined: 2021-12-11 | Last Payment: 2025-12-02 | Payment Status: On-time autopay | Claims: 4 total (3 approved, 1 denied for policy exclusion) | Support Notes: High-value client, consider loyalty discount.",
    
    "Customer ID: CUST005 | Name: Emily Chen | Age: 30 | Subscription Tier: Basic | Joined: 2025-02-15 | Last Payment: 2025-11-30 | Payment Status: Pending for 3 days | Claims: 1 (Minor cosmetic repair Aug 2025 - approved) | Support Notes: Reached out about billing confusion.",
    
    "Customer ID: CUST006 | Name: Robert Davis | Age: 50 | Subscription Tier: Standard | Joined: 2023-07-01 | Last Payment: 2025-12-01 | Payment Status: On-time | Claims: 3 (Home appliance coverage) | Support Notes: Calls often, prefers phone contact.",
    
    "Customer ID: CUST007 | Name: Sofia Garcia | Age: 22 | Subscription Tier: Student | Joined: 2025-09-10 | Last Payment: 2025-11-28 | Payment Status: On-time | Claims: 0 | Support Notes: Limited budget, might downgrade.",
    
    "Customer ID: CUST008 | Name: Daniel Kim | Age: 41 | Subscription Tier: Premium Plus | Joined: 2020-04-05 | Last Payment: 2025-12-05 | Payment Status: On-time | Claims: 6 (High incident average; flagged for review) | Support Notes: Very active user, expects priority support."
]

In [5]:
def tokenize(text):
    #lowercase
    text = text.lower()
    #replace every character that is not a–z, 0–9, or a space with a space " "
    for character in string.punctuation:
        text = text.replace(character," ")
    #splits on whitespaces to isolate words
    tokens = text.split()
    return tokens

In [6]:
#example
a = "I love chocolate ice-cream !"
print(tokenize(a))

['i', 'love', 'chocolate', 'ice', 'cream']


In [7]:
vocab = {}
for doc in documents:
    for token in tokenize(doc):
        if token not in vocab:
            vocab[token] = len(vocab)

V → number of unique words (vector size)

N → number of documents

In [8]:
V = len(vocab)
N = len(documents)

We build a documents matrix where:

- each row is a document

- each column is a word

- each entry is the count of that word in that document

that is a classic bag-of-words representation

In [9]:
doc_matrix = np.zeros((N,V),dtype=np.float32)

In [10]:
for i, doc in enumerate(documents):
    for token in tokenize(doc):
        j = vocab.get(token)
        if j is not None:
            doc_matrix[i,j] += 1.0

For retrieval we use cosine similarity.

- First, we vectorize the question exactly like a document.
    
    vectorized_question is a bag-of-words vector.

- we then compute cosine similarity between vectorized q and each document vector

cosine(a,b) = (a ⋅ b) / (∥a∥ ∥b∥)

Here we use k=1 because customer data is stored in 1 document per customer.

In [1]:
def retrieve_topk(question,k=1):
    # 1: vectorize the question
    vectorized_question = np.zeros(V, dtype=np.float32)
    for token in tokenize(question):
        j = vocab.get(token)
        if j is not None:
            vectorized_question[j] += 1.0
    # 2: cosine similarity with docs
    dot = np.dot(doc_matrix,vectorized_question)
    doc_norms = np.linalg.norm(doc_matrix,axis=1)
    q_norm = np.linalg.norm(vectorized_question)

    sim = dot / (doc_norms * q_norm + 1e-8)

    #we get indices of top-k docs (here k=1)
    indices = np.argsort(-sim)[:k]

    results = []
    for indice in indices:
        results.append((float(sim[indice]), documents[indice]))
    return results

In [11]:
api_key = os.getenv("MISTRAL_API_KEY", "")
if not api_key:
    raise RuntimeError("Set MISTRAL_API_KEY env variable first.")

In [12]:
client = Mistral(api_key=api_key)

In [2]:
def rag_answer(question, k=1, max_tokens=256):
    # 1: we build the context using the retrieved_topk function
    retrieved = retrieve_topk(question, k=k)

    context_string = [f"{doc}" for score, doc in retrieved if score > 0]
    if not context_string:
        return "I could not find anything relevant in the knowledge base."

    context = "\n".join(context_string)

    # 2: we give input to the model so that it only uses our context
    system_message = "You are a helpful assistant. Use ONLY the information in the context. If the information in the context is not enough, reply that YOU DON'T KNOW"

    user_message = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer clearly."

    # 3: we run mistral ai small
    resp = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ],
    max_tokens=max_tokens,
    temperature=0.2,
)

    return resp.choices[0].message.content


To illustrate our RAG architecture, we use it to assist a junior salesperson. It needs to quickly understand the complex situation that is James Turner subscription situation and choose a strategic call approach.

In [15]:
q = "Hey I am a new sales person I want to understand James commercial history in order to prepare my incoming call with him. What should I offer him?"
print("QUESTION:", q)

print("\nRETRIEVED:")
for score, doc in retrieve_topk(q, k=1):
    print(f"[sim={score:.3f}] {doc}")

print("\nANSWER:")
print(rag_answer(q, k=1))

QUESTION: Hey I am a new sales person I want to understand James commercial history in order to prepare my incoming call with him. What should I offer him?

RETRIEVED:
[sim=0.262] Customer ID: CUST002 | Name: James Turner | Age: 45 | Subscription Tier: Basic | Joined: 2022-10-02 | Last Payment: 2025-11-15 | Payment Status: Late last month but now settled | Claims: 0 | Support Notes: ((2023-06-02) Interested in upgrading next year. ; (2024-05-12) Declined sales team special Premium tier offer. ; (2024-05-17) Complained about Basic tier limitations ; (2025-01-06) Asked for a Premium Tier discount, refused by sales team 

ANSWER:
Based on the context, James Turner has shown interest in upgrading his subscription tier but has also declined a special offer and asked for a discount which was refused. He has complained about the limitations of the Basic tier. Given this information, you might want to:

1. Acknowledge his previous interest in upgrading.
2. Address his concerns about the Basic 